# Chlorophyll-a Estimation — CHL69 (Gitelson 1992)
**Algorithm:** Clear OWT. Gitelson 1992. R²=0.79

Formula: `CHL69 = 5956 × (R709 − (R665 + R754) / 2) + 3.84`

Requires reflectance bands at **665 nm**, **709 nm**, and **754 nm**.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# ── Load data ──────────────────────────────────────────────────────────────
CSV_PATH = "2023_May_Month_data.csv"  # adjust path if needed

df = pd.read_csv(CSV_PATH)
print(f"Loaded {df.shape[0]} rows × {df.shape[1]} columns")
df.head(3)

Loaded 908 rows × 753 columns


,DateTime,quality_flag,is_outlier,stability,351,352,353,354,355,356,...,1090,1091,1092,1093,1094,1095,1096,1097,1098,1099
0,2023-05-06 05:04:44,0,False,stable,0.003040,0.002954,0.003016,0.002876,0.003026,0.003016,...,0.000616,0.002051,0.001096,0.000318,0.002261,0.002224,0.001269,0.001046,0.002040,0.001608
1,2023-05-06 05:19:26,0,False,stable,0.003048,0.003080,0.003274,0.003171,0.003195,0.003022,...,0.001283,0.001710,0.000640,0.001475,0.003484,0.001038,0.001747,0.000550,0.002243,0.001518
2,2023-05-06 06:19:28,0,False,stable,0.006696,0.006689,0.006614,0.006424,0.006420,0.006562,...,0.001981,0.001860,0.003117,0.000732,0.000809,0.002579,0.001353,0.001858,0.001714,0.001668


In [4]:
# ── Verify required bands are present ──────────────────────────────────────
required = [665, 709, 754]
missing = [str(w) for w in required if str(w) not in df.columns]

if missing:
    raise ValueError(f"Missing wavelength columns: {missing}")
else:
    print("All required bands found:", required)

All required bands found: [665, 709, 754]


In [5]:
# ── Helper & CHL69 function ────────────────────────────────────────────────
def g(row, wl):
    """Get reflectance value for a given wavelength (nm)."""
    col = str(wl)
    return row[col] if col in row.index else np.nan


def chl_CHL69(row):
    """Clear OWT. Gitelson 1992. R²=0.79."""
    r709, r665, r754 = g(row, 709), g(row, 665), g(row, 754)
    if any(np.isnan(v) for v in [r709, r665, r754]):
        return np.nan
    return 5956.0 * (r709 - (r665 + r754) / 2.0) + 3.84

In [6]:
# ── Apply to every row ─────────────────────────────────────────────────────
df['CHL69'] = df.apply(chl_CHL69, axis=1)

print("CHL69 summary (μg/L):")
print(df['CHL69'].describe().round(3))
print(f"\nNaN count: {df['CHL69'].isna().sum()}")

CHL69 summary (μg/L):
count    908.000
mean      30.405
std       15.576
min       12.154
25%       19.157
50%       25.254
75%       36.329
max       90.119
Name: CHL69, dtype: float64

NaN count: 0


C:\Users\sksus\AppData\Local\Temp\ipykernel_31196\2931158757.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['CHL69'] = df.apply(chl_CHL69, axis=1)


In [7]:
# ── Preview results ────────────────────────────────────────────────────────
meta_cols = [c for c in ['DateTime', 'quality_flag', 'is_outlier', 'stability'] if c in df.columns]
df[meta_cols + ['CHL69']].head(10)

,DateTime,quality_flag,is_outlier,stability,CHL69
0,2023-05-06 05:04:44,0,False,stable,33.563543
1,2023-05-06 05:19:26,0,False,stable,34.114830
2,2023-05-06 06:19:28,0,False,stable,37.199237
3,2023-05-06 06:34:27,0,False,stable,32.148323
4,2023-05-06 06:49:27,0,False,stable,31.831878
5,2023-05-06 07:04:28,0,False,stable,32.714137
6,2023-05-06 07:34:26,0,False,stable,33.361924
7,2023-05-06 07:49:27,0,False,stable,30.649519
8,2023-05-06 09:19:26,0,False,stable,32.067208
9,2023-05-06 09:34:28,262144,False,stable,32.740924


In [10]:
# ── Save results ───────────────────────────────────────────────────────────
out_cols = meta_cols + ['CHL69']
output_path = "2023_May_CHL69_results.csv"
df[out_cols].to_csv(output_path, index=False)
print(f"Saved → {output_path}")

Saved → 2023_May_CHL69_results.csv
